# 20260716_EDA_2024_전국_시도분리정제

- 작성자: 이정연
- #5 이슈 참고 
- 공통 함수는 일단 따로 안만들고, 이 노트북에서 진행해보고 결정해서 유틸로 분리하는 방식으로 진행

In [1]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [2]:
# 데이터 로드
file_2024 = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2024년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획)_칼럼정렬.xlsx"
)
df_raw = pd.read_excel(file_2024, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

(7340, 12)


,지역,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,3906937,3952421.7,45485,1,NaN,1,3,5,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),공통,3260099,3058321,201778,7,NaN,1,3,6,데이터행
2,서울,저출생 극복 지역네트워크 구축사업 지원,공통,54,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행
3,서울,입양아동 가족지원,공통,4002,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행
4,서울,국공립어린이집 등 보육서비스\n수준제고,공통,459506,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행
5,서울,어린이집 이용 영유아 무상보육 확대,공통,554805,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행
6,서울,초등돌봄 공적\n인프라 확충,공통,1141,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행
7,서울,육아종합지원센터 운영,공통,2244,2476,-232,-9,"지원대상 : 어린이집 및 영유아 양육 가정 지원내용\n어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등\n가정양육 지원사업 : 장난감도서관 운영, 시\n간제보육 관리등",1,3,12,데이터행
8,서울,가정양육수당 지원,공통,44151,55556,-11405,-21,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용:월10만원(장애아동35개월미만20만원) 지원,1,3,13,데이터행
9,서울,부모급여,공통,688871,388889,299982,77,"지원대상 : 0~1세 아동\n지원내용:0세100만원(월),1세50만원(월)지원",1,3,14,데이터행


In [3]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (7340, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2024년 예산    object
2023년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.000
사업분류재정구분    0.023
2024년 예산    0.002
2023년 예산    0.004
증감액         0.002
비율          0.005
주요내용        0.041
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1326
경남     725
경북     705
부산     694
충북     512
강원     507
전남     479
전북     430
인천     339
울산     313
대구     264
서울     247
광주     211
대전     173
제주     167
세종     131
충남     117
Name: count, dtype: int64


In [4]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 507
경기 1326
경남 725
경북 705
광주 211
대구 264
대전 173
부산 694
서울 247
세종 131
울산 313
인천 339
전남 479
전북 430
제주 167
충남 117
충북 512


In [5]:
# 서울만 따로 확인 - #3 산출물 검증
df_seoul = sido_dfs["서울"]

print(df_seoul.shape)
df_seoul.head(10)

(247, 12)


,지역,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,3906937,3952421.7,45485,1,NaN,1,3,5,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),공통,3260099,3058321,201778,7,NaN,1,3,6,데이터행
2,서울,저출생 극복 지역네트워크 구축사업 지원,공통,54,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행
3,서울,입양아동 가족지원,공통,4002,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행
4,서울,국공립어린이집 등 보육서비스\n수준제고,공통,459506,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행
5,서울,어린이집 이용 영유아 무상보육 확대,공통,554805,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행
6,서울,초등돌봄 공적\n인프라 확충,공통,1141,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행
7,서울,육아종합지원센터 운영,공통,2244,2476,-232,-9,"지원대상 : 어린이집 및 영유아 양육 가정 지원내용\n어린이집 지원사업 : 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등\n가정양육 지원사업 : 장난감도서관 운영, 시\n간제보육 관리등",1,3,12,데이터행
8,서울,가정양육수당 지원,공통,44151,55556,-11405,-21,지원대상 : 시설 미이용 24개월~86개월 미만 아동 지원내용:월10만원(장애아동35개월미만20만원) 지원,1,3,13,데이터행
9,서울,부모급여,공통,688871,388889,299982,77,"지원대상 : 0~1세 아동\n지원내용:0세100만원(월),1세50만원(월)지원",1,3,14,데이터행


In [6]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw["2024년 예산"] = pd.to_numeric(
    df_raw["2024년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
)

-> #3의 결과와 정확하게 일치하는 것 확인

-> 이 노트북에서 서울은 따로 데이터셋 정제 불필요

In [7]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계,헤더반복
지역,,,,
강원,2,496,8,1
경기,2,1315,8,1
경남,2,714,8,1
경북,2,694,8,1
광주,2,200,8,1
대구,2,253,8,1
대전,2,162,8,1
부산,2,683,8,1
서울,2,237,8,0


-> 17개 시도 모두 헤더반복 노이즈 0건 확인 

-> 정리본_자동 sheet에서 정제된 것 확인 

-> 대분류_소계도 2개로 일관적임을 확인

-> 경기 경남에서 중분류_소계가 9건이라 확인 필요

In [8]:
# 경기 중분류_소계 행 확인
df_gyeonggi = sido_dfs["경기"]
df_gyeonggi["사업행구분"] = df_gyeonggi["세부사업명"].apply(classify_row)
print("[경기] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeonggi[df_gyeonggi["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 경남도 동일하게 확인
df_gyeongnam = sido_dfs["경남"]
df_gyeongnam["사업행구분"] = df_gyeongnam["세부사업명"].apply(classify_row)
print("[경남] 중분류_소계 행 확인")
print("==================================================")
print(df_gyeongnam[df_gyeongnam["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

[경기] 중분류_소계 행 확인
2374      1. 함께 일하고 함께 돌보는 사회(공통)
2414       2. 건강하고 능동적인\n고령사회(공통)
2423    3. 모두의 역량이 고루 발휘되는 사회(공통)
2436       4.인구구조\n변화에 대한\n적응(공통)
2443      1. 함께 일하고 함께 돌보는 사회(자체)
3030     2. 건강하고 능동적인 고령사회 구축(자체)
3214    3. 모두의 역량이 고루 발휘되는 사회(자체)
3519         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


[경남] 중분류_소계 행 확인
6450      1. 함께 일하고 함께 돌보는 사회(공통)
6503       2. 건강하고 능동적인\n고령사회(공통)
6515    3. 모두의 역량이 고루 발휘되는 사회(공통)
6526         4.인구구조 변화에 대한 적응(공통)
6531      1. 함께 일하고 함께 돌보는 사회(자체)
6813    2. 건강하고 능동적인 고령사회\n구축(자체)
6929    3. 모두의 역량이 고루 발휘되는 사회(자체)
7079         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


-> 진짜 중분류_소계 행들을 보면 뒤에 (공통) / (자체) 가 오는 것을 확인할 수 있음. 

-> 이  끝부분 조건 추가하면 오분류 걸러내기 가능함. 

-> classify_row 함수 수정 

-> 조건 추가하니까 해결됨

In [9]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

(7154, 15)
['세부사업명', '사업분류재정구분', '2024년 예산', '2023년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 극복 지역네트워크 구축사업 지원,공통,54.0,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정)\n지원내용:서울100인의아빠단,1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,입양아동 가족지원,공통,4002.0,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,국공립어린이집 등 보육서비스\n수준제고,공통,459506.0,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,어린이집 이용 영유아 무상보육 확대,공통,554805.0,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,초등돌봄 공적\n인프라 확충,공통,1141.0,5087,-3946,-78,지원대상 : 6~12세 아동\n지원내용:돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [10]:
mask_non_numeric = (
    pd.to_numeric(df_leaf["2024년 예산"], errors="coerce").isna() & df_leaf["2024년 예산"].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", "2024년 예산"]])

,지역,세부사업명,2024년 예산


In [11]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])["2024년 예산"]
    .sum()
    .reset_index()
    .rename(columns={"2024년 예산": "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", "2024년 예산"]
].rename(columns={"세부사업명": "중분류", "2024년 예산": "원본_소계값"})

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["결과"].value_counts()

결과
일치     114
불일치     22
Name: count, dtype: int64

In [12]:
qa.head()

,지역,대분류,중분류,원본_소계값,leaf_합계,결과
0,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),3260099.0,3260099.0,일치
1,서울,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),84128.0,84128.0,일치
2,서울,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는\n사회(공통),458096.0,458096.0,일치
3,서울,Ⅰ. 공통사업,4.인구구조 변화에 대한 적응(공통),104614.0,104614.0,일치
4,서울,Ⅱ. 지자체\n자체사업,1. 함께 일하고 함께 돌보는 사회(자체),483598.0,483598.0,일치


In [13]:
qa["차이"] = qa["leaf_합계"] - qa["원본_소계값"]

# 결과 컬럼을 항상 채워서, 저장 파일이 비어있어도 "검증은 했다"는 게 남도록 함
qa["결과"] = qa["차이"].abs().le(0.01).map({True: "일치", False: "불일치"})

qa_mismatch = qa[qa["결과"] == "불일치"]
print(f"검증 대상: {len(qa)}건 / 불일치: {len(qa_mismatch)}건")

검증 대상: 136건 / 불일치: 22건


In [14]:
display(qa_mismatch[["지역", "대분류", "중분류", "원본_소계값", "leaf_합계", "차이"]])

,지역,대분류,중분류,원본_소계값,leaf_합계,차이
12,부산,Ⅱ. 지자체\n자체사업,1. 함께 일하고\n함께 돌보는\n사회(자체),511612.0,511613.0,1.0
14,부산,Ⅱ. 지자체\n자체사업,3. 모두의 역량이 고루 발휘되는\n사회(자체),58329.0,58331.0,2.0
35,광주,Ⅰ. 공통사업,4.인구구조 변화에 대한 적응(공통),0.0,NaN,NaN
68,경기,Ⅱ. 지자체\n자체사업,1. 함께 일하고 함께 돌보는 사회(자체),1019222.0,1019224.0,2.0
70,경기,Ⅱ. 지자체\n자체사업,3. 모두의 역량이 고루 발휘되는 사회(자체),425962.0,425965.0,3.0
84,충북,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),121958.0,121959.0,1.0
85,충북,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인 고령사회\n구축(자체),36615.0,36616.0,1.0
87,충북,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),116980.0,116979.0,-1.0
88,충남,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),522929.0,411451.0,-111478.0
89,충남,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),617588.0,21136.0,-596452.0


# 충남 QA 불일치 원인 확인 - 원본 Table 1 대조

`정리본_자동`에서 충남만 QA 불일치가 크게 나온 원인이 `정리본_자동` 변환 과정의 데이터 손실인지, 원본 문서 자체의 문제인지 확인한다.

`원본행` 컬럼은 `Table 1` 시트의 1-indexed 엑셀 행 번호를 그대로 가리키므로, 이 값으로 원본 위치를 바로 대조할 수 있다.

In [15]:
df_chungnam_raw = sido_dfs["충남"]
mask_nan = pd.to_numeric(
    df_chungnam_raw["2024년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
).isna()

display(df_chungnam_raw.loc[mask_nan, ["세부사업명", "2024년 예산"]])

,세부사업명,2024년 예산
4717,붙임 (충청남도 충청남도교육청),붙임 (충청남도 충청남도교육청)


In [16]:
df_chungnam = df_labeled[df_labeled["지역"] == "충남"]
print(df_chungnam["사업행구분"].value_counts())
display(df_chungnam[df_chungnam["사업행구분"] == "중분류_소계"]["세부사업명"])

사업행구분
세부사업      106
중분류_소계      8
대분류_소계      2
헤더반복        1
Name: count, dtype: int64


4719      1. 함께 일하고 함께 돌보는 사회(공통)
4759       2. 건강하고 능동적인\n고령사회(공통)
4772    3. 모두의 역량이 고루 발휘되는 사회(공통)
4773         4.인구구조 변화에 대한 적응(공통)
4779    1. 함께 일하고\n함께 돌보는\n사회(자체)
4799    2. 건강하고 능동적인 고령사회\n구축(자체)
4815    3. 모두의 역량이 고루 발휘되는 사회(자체)
4827         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str

In [17]:
df_chungnam_check = df_labeled[df_labeled["지역"] == "충남"][
    ["원본행", "세부사업명", "사업행구분", "대분류", "중분류"]
]

display(df_chungnam_check.head(30))

,원본행,세부사업명,사업행구분,대분류,중분류
4717,5294,붙임 (충청남도 충청남도교육청),헤더반복,NaN,NaN
4718,5298,Ⅰ. 공통사업,대분류_소계,Ⅰ. 공통사업,NaN
4719,5299,1. 함께 일하고 함께 돌보는 사회(공통),중분류_소계,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4720,5300,이동식놀이교실\n운영,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4721,5301,시간제 보육사업 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4722,5302,영유아보육료,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4723,5303,가정양육수당,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4724,5304,부모급여,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4725,5305,여성장애인 출산비용 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)
4726,5306,분만취약지 지원,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통)


In [18]:
# 원본 Table 1 로드
file_2024_original = (
    DATA_DIR / "세부사업표추출_2024년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획).xlsx"
)
df_table1 = pd.read_excel(file_2024_original, sheet_name="Table 1", header=None)


def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변을 보여줌"""
    start, end = center_excel_row - window, center_excel_row + window
    print(f"--- {label} (Table1 엑셀행 {start}~{end}) ---")
    # 1,3열은 병합셀로 생긴 빈 열이라 제외, 나머지는 알아보기 쉬운 이름으로 표시
    view = df_table1.iloc[start - 1 : end, [0, 2, 4]].copy()
    view.columns = ["세부사업명", "공통/자체", "예산"]
    display(view)


# 충남 "1. 함께 일하고..." 중분류 안에서 원본행이 연속되지 않는(결번) 구간을 직접 탐지
df_chungnam_cat1 = df_chungnam_check[
    df_chungnam_check["중분류"] == "1. 함께 일하고 함께 돌보는 사회(공통)"
]
원본행_목록 = df_chungnam_cat1["원본행"].tolist()
결번_구간 = [(a, b) for a, b in zip(원본행_목록, 원본행_목록[1:]) if b - a > 1]
print("결번 구간:", 결번_구간)

for 시작, 끝 in 결번_구간:
    show_table1_around((시작 + 끝) // 2, window=(끝 - 시작), label=f"{시작}~{끝} 결번 구간")

# "3. 모두의 역량이 고루 발휘되는 사회(공통)" 소계 행의 실제 원본행을 가져와 그 주변 확인
소계_행 = df_labeled[
    (df_labeled["지역"] == "충남")
    & (df_labeled["세부사업명"].str.contains("모두의 역량", na=False))
    & (df_labeled["사업행구분"] == "중분류_소계")
]
show_table1_around(int(소계_행["원본행"].iloc[0]), window=3, label="3.모두의역량 소계 전후")

결번 구간: [(5312, 5315), (5333, 5336)]
--- 5312~5315 결번 구간 (Table1 엑셀행 5310~5316) ---


,세부사업명,공통/자체,예산
5309,청소년산모 임신출산 의료비\n지원,공통,26
5310,고위험임산부\n의료비 지원,공통,788
5311,기저귀 및 조제분유 지원,공통,4194
5312,세부사업명,사업 분류,2024년 예산
5313,NaN,공통/ 자체,총예산(a)
5314,첫만남이용권 지원,공통,22874
5315,생애초기건강관리 시범사업,공통,6896


--- 5333~5336 결번 구간 (Table1 엑셀행 5331~5337) ---


,세부사업명,공통/자체,예산
5330,저소득층 기저귀\n조제분유 지원,공통,265
5331,산모신생아\n건강관리 지원사업,공통,1446
5332,산후조리도우미 본인부담금 지원사업,공통,335
5333,세부사업명,사업 분류,2024년 예산
5334,NaN,공통/ 자체,총예산(a)
5335,다자녀 맘 산후\n건강관리 지원,공통,76
5336,난임부부 시술비\n지원,공통,630


--- 3.모두의역량 소계 전후 (Table1 엑셀행 5355~5361) ---


,세부사업명,공통/자체,예산
5354,희망소리 찾기 사업(보청기 지원사업),공통,8
5355,세부사업명,사업 분류,2024년 예산
5356,NaN,공통/ 자체,총예산(a)
5357,3. 모두의 역량이 고루 발휘되는 사회(공통),NaN,19928
5358,4.인구구조 변화에 대한 적응(공통),NaN,17795
5359,한부모가족자녀의\n안정적 성장 지원,공통,764
5360,한부모가족\n생활안정 지원,공통,495


# 증감액 부호 소실 문제 확인

팀원 지적: 충남 "3. 모두의 역량이 고루 발휘되는 사회(공통)"는 2024년 예산(19,928)이 2023년 예산(42,867.5)보다 줄었으니 증감액·증감율이 음수여야 하는데, 원본에 양수로 찍혀있다. 이게 충남만의 문제인지, 다른 시도에도 있는지 확인한다.

방법: 세부사업별로 `2024년 예산 - 2023년 예산`을 직접 계산해서, 그 결과가 음수(예산 감소)인 행들만 골라 `증감액` 컬럼이 실제로 음수로 찍혀있는지 대조한다.

In [19]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2024년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


# 이전 셀 실행 여부와 무관하게 독립적으로 동작하도록 2024년 예산도 여기서 다시 변환
df_raw["2024년 예산_num"] = to_numeric_budget(df_raw["2024년 예산"])
df_raw["2023년 예산_num"] = to_numeric_budget(df_raw["2023년 예산"])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["계산된_증감액"] = df_raw["2024년 예산_num"] - df_raw["2023년 예산_num"]

# 실제로 예산이 감소한 행(계산된 증감액 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["계산된_증감액"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

예산 감소 행 수: 2193
그중 증감액이 양수로 찍힌(부호 소실) 행 수: 982


In [20]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

지역
인천    100.0
경남    100.0
충남    100.0
대구    100.0
대전    100.0
부산    100.0
전북    100.0
전남     83.3
제주     64.2
서울     61.6
광주      7.9
충북      1.3
울산      0.0
세종      0.0
경기      0.0
경북      0.0
강원      0.0
Name: 부호_소실, dtype: float64

# 증감액 / 비율 직접 재계산
- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음. 
- 따라서 직접 재계산

In [21]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환 (df_raw의 2024년 예산 변환과 동일한 규칙)"""
    return pd.to_numeric(series.astype(str).str.replace(",", "").replace("-", 0), errors="coerce")


df_leaf["2023년_예산_num"] = to_numeric_budget(df_leaf["2023년 예산"])
df_leaf["2024년_예산_num"] = to_numeric_budget(df_leaf["2024년 예산"])
df_leaf["증감액_재계산"] = df_leaf["2024년_예산_num"] - df_leaf["2023년_예산_num"]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf["2023년_예산_num"].replace(0, np.nan) * 100
).round(1)

df_leaf[
    ["세부사업명", "2024년_예산_num", "2023년_예산_num", "증감액_재계산", "증감율_재계산"]
].head(10)

,세부사업명,2024년_예산_num,2023년_예산_num,증감액_재계산,증감율_재계산
2,저출생 극복 지역네트워크 구축사업 지원,54.0,108.0,-54.0,-50.0
3,입양아동 가족지원,4002.0,4345.0,-343.0,-7.9
4,국공립어린이집 등 보육서비스\n수준제고,459506.0,450562.0,8944.0,2.0
5,어린이집 이용 영유아 무상보육 확대,554805.0,654969.0,-100164.0,-15.3
6,초등돌봄 공적\n인프라 확충,1141.0,5087.0,-3946.0,-77.6
7,육아종합지원센터 운영,2244.0,2476.0,-232.0,-9.4
8,가정양육수당 지원,44151.0,55556.0,-11405.0,-20.5
9,부모급여,688871.0,388889.0,299982.0,77.1
10,공동육아나눔터,2365.0,2136.0,229.0,10.7
11,아이돌봄서비스 확충 및 내실화,96810.0,76220.0,20590.0,27.0


# 최종 스키마

In [22]:
# 텍스트 정리
def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""
    return series.apply(lambda x: re.sub(r"\s+", " ", str(x)).strip() if pd.notna(x) else x)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])

df_leaf.head()

,세부사업명,사업분류재정구분,2024년 예산,2023년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역,2023년_예산_num,2024년_예산_num,증감액_재계산,증감율_재계산
2,저출생 극복 지역네트워크 구축사업 지원,공통,54.0,108,-54,-50,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용:서울100인의아빠단,1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,108.0,54.0,-54.0,-50.0
3,입양아동 가족지원,공통,4002.0,4345,-343,-8,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,4345.0,4002.0,-343.0,-7.9
4,국공립어린이집 등 보육서비스 수준제고,공통,459506.0,450562,8944,2,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,450562.0,459506.0,8944.0,2.0
5,어린이집 이용 영유아 무상보육 확대,공통,554805.0,654969,-100164,-15,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,654969.0,554805.0,-100164.0,-15.3
6,초등돌봄 공적 인프라 확충,공통,1141.0,5087,-3946,-78,지원대상 : 6~12세 아동 지원내용:돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,5087.0,1141.0,-3946.0,-77.6


In [30]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            "2024년_예산_num": "당해예산",
            "2023년_예산_num": "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=2024)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

print(df_leaf_final.shape)
display(df_leaf_final.head())

(7154, 12)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행
2,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 극복 지역네트워크 구축사업 지원,지원대상 : 서울시민(3~7세 자녀가 있는 가정) 지원내용:서울100인의아빠단,54.0,108.0,-54.0,-50.0,7
3,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,입양아동 가족지원,"지원대상 : 입양특례법에 의한 18세미만 입양 아동 및 그 가정 지원내용:입양아동양육보조금,양육수당,입양비용등",4002.0,4345.0,-343.0,-7.9,8
4,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 등 보육서비스 수준제고,지원대상 : 국공립 어린이집 등 지원내용:보육교직원인건비지원,459506.0,450562.0,8944.0,2.0,9
5,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,어린이집 이용 영유아 무상보육 확대,지원대상 : 만0~2세 영아 지원내용:어린이집이용영아대상바우처지원,554805.0,654969.0,-100164.0,-15.3,10
6,2024,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,초등돌봄 공적 인프라 확충,지원대상 : 6~12세 아동 지원내용:돌봄서비스제공,1141.0,5087.0,-3946.0,-77.6,11


In [26]:
print(df_leaf_final.columns.tolist())
print(df_leaf_final.columns.duplicated().any())
print(df_leaf_final.columns[df_leaf_final.columns.duplicated()])

['세부사업명', '사업분류재정구분', '2024년 예산', '2023년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역', '전년도예산', '당해예산', '증감액', '증감율', '연도']
True
Index(['증감액'], dtype='str')


In [31]:
year = 2024

for sido in df_leaf_final["지역"].unique():
    sido_leaf = df_leaf_final[df_leaf_final["지역"] == sido].drop(columns=["지역"])
    sido_labeled = df_labeled[df_labeled["지역"] == sido].drop(columns=["지역"])

    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)

    sido_leaf.to_csv(
        sido_dir / f"{year}_{sido}_세부사업_정제.csv", index=False, encoding="utf-8-sig"
    )

# qa 결과는 전체 17개 시도 한 파일로 저장
qa.to_csv(REPORTS_DIR / f"{year}_전국_qa_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

저장 완료:  17 개 시도
